# Churn advanced models

## 01. Executive overview

Can nonlinear models make a retention review queue more selective? The approved menu
compares a prior dummy, two L2 logistic fits, two Random Forest configurations and two
histogram gradient boosting configurations. All use the same forward, customer-disjoint
development protocol and training-only transformations.

The nominated boosting candidate led validation average precision (AP) by roughly
0.022 percentage points over the dummy in the retained run. Its low ROC-AUC and 695
false alerts among 703 alerts give little evidence of useful prioritisation. The
eight positive validation accounts limit comparisons. The seed-2027 reserve remains
**unscored**. These are development results from a fictional subscription population.

## 02. Baseline reference

The checksummed reports bind both analyses to one design and the same baseline artefact.
This notebook reads them without fitting. See the [baseline notebook](05_churn_baseline_models.ipynb)
for target, feature and population reconciliation, and the
[evaluation protocol](../../docs/05_ml/phase06_churn_evaluation_protocol.md) for the fixed rules.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.ticker import PercentFormatter

from nowa_customer_intelligence.models.churn.research import DEFAULT_WORKSPACE, load_reports

repository = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
results_dir = Path(os.environ.get("NCI_CHURN_RESULTS", repository / DEFAULT_WORKSPACE / "results"))
baseline, advanced = load_reports(results_dir)
design = baseline["design"]
plt.style.use("seaborn-v0_8-whitegrid")
palette = ["#65758b", "#245293", "#138577", "#854ba5", "#ae5c27", "#a23c55", "#151e37"]
plt.rcParams.update({"figure.dpi": 120, "axes.titleweight": "bold", "font.size": 10})


def metric_table(candidates):
    rows = []
    for item in candidates:
        row = item["validation"]
        rows.append(
            {
                "Model": item["name"],
                "Threshold": item["threshold"],
                "AP": row["average_precision"],
                "ROC-AUC": row["roc_auc"],
                "Precision": row["precision"],
                "Recall": row["recall"],
                "F1": row["f1"],
                "Brier": row["brier"],
                "Positives": row["positives"],
                "TN": row["tn"],
                "FP": row["fp"],
                "FN": row["fn"],
                "TP": row["tp"],
            }
        )
    return pd.DataFrame(rows).set_index("Model")


def show():
    plt.tight_layout()
    plt.show()
    plt.close("all")


references = baseline["candidates"]
advanced_models = advanced["candidates"]
comparison = references + advanced_models
winner = next(item for item in comparison if item["name"] == advanced["selected"])
metrics = metric_table(comparison)


def operational_table(candidates):
    rows = []
    for candidate in candidates:
        m = candidate["validation"]
        alerts = m["tp"] + m["fp"]
        assert m["tn"] + alerts + m["fn"] == m["rows"]
        assert m["tp"] + m["fn"] == m["positives"]
        assert np.isclose(m["precision"], m["tp"] / alerts if alerts else 0)
        assert np.isclose(m["recall"], m["tp"] / m["positives"] if m["positives"] else 0)
        rows.append(
            {
                "Model": candidate["name"],
                "Rows": m["rows"],
                "Alerts": alerts,
                "Alert rate": alerts / m["rows"],
                "True alerts": m["tp"],
                "False alerts": m["fp"],
                "Missed churn": m["fn"],
                "False discovery": m["fp"] / alerts if alerts else np.nan,
                "Precision": m["precision"],
                "Recall": m["recall"],
            }
        )
    return pd.DataFrame(rows).set_index("Model")


def plot_alerts(table):
    fig, ax = plt.subplots(figsize=(10, 1.8 + 0.52 * len(table)))
    columns = ["False alerts", "True alerts"]
    table[columns].plot.barh(stacked=True, ax=ax, color=["#a74932", "#167568"], width=0.58)
    for index, (alerts, rate) in enumerate(zip(table["Alerts"], table["Alert rate"], strict=True)):
        ax.text(
            alerts + table["Rows"].max() * 0.016, index, f"{alerts:,} ({rate:.1%})", va="center"
        )
    ax.set(
        title="Validation review queue at each frozen threshold",
        xlabel="Alerted episode observations",
        ylabel="",
        xlim=(0, table["Rows"].max() * 1.22),
    )
    ax.invert_yaxis()
    ax.grid(False, axis="y")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False)
    show()


def cohort_table(records):
    table = pd.DataFrame(records).rename(
        columns={
            "cohort": "Cohort",
            "level": "Group",
            "rows": "Rows",
            "positives": "Positive rows",
            "positive_accounts": "Positive accounts",
            "tp": "True alerts",
            "fp": "False alerts",
            "fn": "Missed churn",
        }
    )
    table["Alert rate"] = (table["True alerts"] + table["False alerts"]) / table["Rows"]
    table["False alerts / all rows"] = table["False alerts"] / table["Rows"]
    return table.drop(columns="recall")


def plot_confusion(candidates):
    fig, axes = plt.subplots(1, len(candidates), figsize=(4 * len(candidates), 3.5), squeeze=False)
    for ax, candidate in zip(axes[0], candidates, strict=True):
        m = candidate["validation"]
        matrix = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])
        # Shade within observed classes so rare positive cases remain visible.
        fractions = matrix / matrix.sum(axis=1, keepdims=True)
        ax.imshow(fractions, cmap="Blues", vmin=0, vmax=1)
        ax.grid(False)
        for (y, x), value in np.ndenumerate(matrix):
            fraction = fractions[y, x]
            ax.text(
                x,
                y,
                f"{value:,}\n{fraction:.1%}",
                ha="center",
                va="center",
                color="white" if fraction > 0.56 else "#131f31",
            )
        ax.set(
            xticks=[0, 1],
            xticklabels=["No alert", "Alert"],
            yticks=[0, 1],
            yticklabels=["No churn", "Churn"],
            xlabel="Frozen decision",
            title=candidate["name"],
            ylabel="Resolved outcome",
        )
    fig.suptitle("Validation confusion matrices · counts and within-class percentages")
    show()


operations = operational_table(comparison)
metrics = metrics.join(operations[["Alerts", "Alert rate"]])
chosen_metrics = winner["validation"]
reference_ap = references[0]["validation"]["average_precision"]
strongest_logistic = next(item for item in references if item["name"] == baseline["selected"])
focused = [references[0], strongest_logistic, winner]


def format_table(frame, formats, na_rep="—"):
    """Format display values while retaining numeric source columns."""
    display_frame = frame.copy()
    for column, pattern in formats.items():
        display_frame[column] = frame[column].map(pattern.format, na_action="ignore").fillna(na_rep)
    return display_frame

## 03. Advanced modelling strategy

Tree splits can represent interactions among billing cadence, episode age and usage.
That flexibility can amplify synthetic timing or rare combinations. Both families use
the same Phase 05 predictors. Preprocessing, weights and estimator parameters come
from the fixed training run. Metadata and future outcome evidence remain outside the
predictor set.

## 04. Random Forest

A Random Forest averages trees trained on bootstrap samples. Depth limits the
complexity of each tree. A minimum leaf size constrains how few examples can determine
a local rule. Both configurations use 120 trees, a 20-row leaf minimum and training-only
balanced subsample weights. Depth 8 improves training AP relative to depth 4 in the
reference run, while validation AP is lower.

In [ ]:
display(metric_table(advanced_models[:2])[["AP", "ROC-AUC", "Precision", "F1"]].round(5))

## 05. Gradient boosting

Histogram gradient boosting adds trees in sequence to reduce training loss. The
learning rate limits each tree's contribution. Leaf limits, minimum leaf support and L2
regularisation constrain fit flexibility. The 15-leaf fit also uses balanced class
weights, so the difference from the seven-leaf fit cannot isolate the effect of weighting
or leaf count.

In [ ]:
display(metric_table(advanced_models[2:])[["AP", "ROC-AUC", "Precision", "F1"]].round(5))

## 06. Fixed hyperparameter comparison

The menu was set before fitting. Early stopping stayed off for boosting to avoid a
random internal split. All configurations share one forward validation window. The
small menu limits search over scarce positive accounts and still leaves selection
uncertainty on the same window.

## 07. Class imbalance

Class weighting changes the contribution of training errors while preserving the
observed population and chronology. The forest weights are computed within each
training bootstrap sample. Boosting weights apply to the balanced configuration.
Neither approach increases the number of independent validation events.

In [ ]:
configuration = pd.DataFrame(
    [{"Model": item["name"], **item["parameters"]} for item in advanced_models]
)
keys = [
    "Model",
    "random_state",
    "class_weight",
    "max_depth",
    "max_leaf_nodes",
    "min_samples_leaf",
    "n_estimators",
    "max_iter",
    "learning_rate",
    "l2_regularization",
    "early_stopping",
]
display(configuration[keys].set_index("Model").fillna("—"))

## 08. Seven-model comparison

Which model improves ranking, score accuracy or decision workload? AP and ROC-AUC measure
ranking. Brier is the mean squared score error. Precision, recall, F1 and queue statistics
describe the frozen decisions. All seven candidates use the same resolved validation rows.

In [ ]:
display(
    format_table(
        metrics,
        {
            "Threshold": "{:.6f}",
            "AP": "{:.5f}",
            "ROC-AUC": "{:.4f}",
            "Brier": "{:.5f}",
            "Precision": "{:.2%}",
            "Recall": "{:.2%}",
            "F1": "{:.4f}",
            "Alert rate": "{:.2%}",
        },
    )
)

validation_prevalence = chosen_metrics["prevalence"]
ap_comparison = pd.DataFrame(
    {
        "AP": metrics["AP"],
        "Prevalence": validation_prevalence,
        "Difference (percentage points)": 100 * (metrics["AP"] - validation_prevalence),
        "Ratio to prevalence": metrics["AP"] / validation_prevalence,
    }
)
display(
    format_table(
        ap_comparison,
        {
            "AP": "{:.6f}",
            "Prevalence": "{:.6f}",
            "Difference (percentage points)": "{:+.4f}",
            "Ratio to prevalence": "{:.3f}×",
        },
    )
)
fig, ax = plt.subplots(figsize=(9, 4.5))
colours = ["#167568" if name == winner["name"] else "#6d7e92" for name in metrics.index]
ax.barh(metrics.index, metrics["AP"], color=colours, height=0.6)
ax.axvline(validation_prevalence, ls="--", color="#20262d", label="Validation prevalence")
ax.set(
    title="Validation average precision against prevalence",
    xlabel="Average precision",
    xlim=(0, max(metrics["AP"].max(), validation_prevalence) * 1.22),
)
ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=2))
ax.invert_yaxis()
ax.legend(loc="lower right")
show()

The AP axis starts at zero. In the retained run, the nominee’s AP was about 1.020 times
prevalence, or just 0.022 percentage points above it. The comparison table shows all
negative deltas as well. With eight positive accounts and no paired performance interval
for the difference, the observed gain offers weak evidence of a ranking improvement.

## 09. Precision–recall and ROC analysis

Focus on the prior reference, the leading logistic fit and the nominated research
candidate. Both panels cover full 0–100% axes. PR shows the share of alerts that are
true and the share of positives detected. ROC shows sensitivity versus the fraction
of negatives alerted. The all-candidate table retains the less favourable results.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.8, 4))
for item, colour, style in zip(
    focused, (palette[0], palette[1], palette[5]), ("-", "--", "-."), strict=True
):
    pr, roc = item["pr"], item["roc"]
    axes[0].step(
        pr["recall"], pr["precision"], where="post", label=item["name"], color=colour, ls=style
    )
    axes[1].plot(roc["false_positive_rate"], roc["true_positive_rate"], color=colour, ls=style)
axes[0].axhline(validation_prevalence, ls=":", color="black", label="Validation prevalence")
axes[1].plot([0, 1], [0, 1], ls=":", color="black", lw=1, label="Uninformative ranking")
axes[0].set(title="Validation precision–recall", xlabel="Recall", ylabel="Precision")
axes[1].set(title="Validation ROC", xlabel="False-positive rate", ylabel="True-positive rate")
for ax in axes:
    ax.set(xlim=(0, 1), ylim=(0, 1))
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.yaxis.set_major_formatter(PercentFormatter(1))
axes[0].legend(fontsize=7.5, loc="upper right")
axes[1].legend(fontsize=8, loc="upper left")
show()

In the reference run, the low ROC-AUC values indicate that positives tended to be
ranked below negatives. An isolated precision step can still affect AP when very few
positives are available. Neither metric demonstrates an effective queue at a frozen
threshold.

## 10. Threshold behaviour

The selected threshold uses the saved validation F1 rule. Other models have different
score scales, especially after weighting. The sweep records precision, recall and F1.
Alert volume is available at the saved threshold from the confusion counts. The
displayed threshold stays fixed.

In [ ]:
sweep = pd.DataFrame(winner["threshold_sweep"])
fig, ax = plt.subplots(figsize=(8, 3.5))
for field, colour in zip(("precision", "recall", "f1"), palette[1:4], strict=True):
    ax.plot(
        sweep["threshold"],
        sweep[field],
        label=field.upper() if field == "f1" else field.title(),
        color=colour,
    )
ax.axvline(winner["threshold"], ls="--", color=palette[0], label="Selected threshold")
ax.set(
    title=f"Validation threshold trade-off · {winner['name']}",
    xlabel="Score threshold",
    ylabel="Metric value",
)
ax.legend()
show()

Validation F1 reflects the same rows that selected each threshold. Treat it as a
descriptive operating point. It gives no deployment-capacity or intervention-cost
evidence.

## 11. Confusion matrices and operational alert burden

The focused matrices use the same actual-class orientation and a shared 0–100%
within-class colour scale. Displayed counts keep the eight positive observations visible
beside the much larger negative class. The table and alert figure retain all seven
candidates, with volume measured in episode observations.

In [ ]:
plot_confusion(focused)

rate_columns = ["Alert rate", "False discovery", "Precision", "Recall"]
display(format_table(operations, dict.fromkeys(rate_columns, "{:.2%}"), na_rep="—"))
plot_alerts(operations)

The reference boosting decision found all eight observed events by flagging **703 of 717
observations (98.0%)**. **695 alerts were false (98.9% of alerts)**. The dummy and every
other fit also reached 100% recall at their saved thresholds. The queue figures show
how little selectivity that recall represents. Alerts can include repeat observations,
so these counts do not translate directly into a count of unique reviews.

## 12. Calibration and score evidence

The reliability panel pairs mean score with observed event rate. The count panel and
table show how much evidence falls in each band. Empty bins have no empirical rate.
All expected score ranges are shown so large miscalibration remains visible.

In [ ]:
calibration_models = [references[0], winner]
curve_colours = [palette[0], palette[5]]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1.2, 1]})
calibration_tables = []
bin_axis = None
for item, colour, marker, offset in zip(
    calibration_models, curve_colours, ("o", "s"), (-0.18, 0.18), strict=True
):
    cal = pd.DataFrame(item["calibration"])
    occupied = cal[cal["count"] > 0]
    axes[0].scatter(
        occupied["mean_score"],
        occupied["observed_rate"],
        label=item["name"],
        color=colour,
        marker=marker,
        s=38,
    )
    bin_axis = cal["bin"]
    axes[1].bar(np.arange(len(cal)) + offset, cal["count"], width=0.35, color=colour)
    calibration_tables.append(cal.assign(Model=item["name"]))
axes[0].plot([0, 1], [0, 1], color="#6a7583", ls="--", lw=1, label="Equal score and event rate")
axes[0].set(
    title="Validation reliability in occupied bins",
    xlabel="Mean score",
    ylabel="Observed event rate",
    xlim=(0, 1),
    ylim=(0, 1),
)
axes[0].legend(fontsize=8)
axes[1].set(
    title="Validation observations per score bin",
    xlabel="Score band",
    ylabel="Episode observations",
    xticks=range(len(bin_axis)),
    xticklabels=bin_axis,
)
axes[1].tick_params(axis="x", labelrotation=24)
for axis in (axes[0].xaxis, axes[0].yaxis):
    axis.set_major_formatter(PercentFormatter(1))
show()
calibration_display = pd.concat(calibration_tables, ignore_index=True).rename(
    columns={
        "bin": "Score band",
        "count": "Rows",
        "positives": "Positive rows",
        "mean_score": "Mean score",
        "observed_rate": "Observed rate",
    }
)
display(
    format_table(
        calibration_display, {"Mean score": "{:.2%}", "Observed rate": "{:.2%}"}, na_rep="—"
    )
)

In the reference run all eight events occurred in the nominated model’s 0–1% bin.
The 35–100% bin had 52 observations and zero events. The small event count limits
precision of calibration estimates, and the observed direction is adverse.
The reports supply aggregate bins and curves. Without a validated observation-level
score ledger, a score-density plot, lift or capacity-specific precision would require
evidence beyond the retained reports.

## 13. Training versus validation

A large gap can reflect model flexibility, temporal shift, differences between account
cohorts and the scarcity of validation positives. The generator’s fixed plan,
reactivation schedule and evidence availability offer plausible mechanisms. The
comparison alone cannot identify their individual effects.

In [ ]:
gap = pd.DataFrame(
    {
        "Model": [c["name"] for c in comparison],
        "Training AP": [c["training"]["average_precision"] for c in comparison],
        "Validation AP": [c["validation"]["average_precision"] for c in comparison],
        "Training prevalence": [c["training"]["prevalence"] for c in comparison],
        "Validation prevalence": [c["validation"]["prevalence"] for c in comparison],
    }
).set_index("Model")
gap["Gap (percentage points)"] = 100 * (gap["Training AP"] - gap["Validation AP"])
display(
    format_table(
        gap,
        {
            "Training AP": "{:.5f}",
            "Validation AP": "{:.5f}",
            "Training prevalence": "{:.2%}",
            "Validation prevalence": "{:.2%}",
            "Gap (percentage points)": "{:.2f}",
        },
    )
)
fig, ax = plt.subplots(figsize=(9, 4.3))
y = np.arange(len(gap))
ax.scatter(gap["Training AP"], y, color="#255889", marker="o", label="Training AP")
ax.scatter(gap["Validation AP"], y, color="#a74932", marker="s", label="Validation AP")
ax.hlines(y, gap["Validation AP"], gap["Training AP"], color="#8994a0", lw=1, zorder=0)
ax.set(
    title="In-sample and forward-validation average precision",
    xlabel="Average precision · shared 0–100% scale",
    yticks=y,
    yticklabels=gap.index,
    xlim=(0, 1),
)
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.invert_yaxis()
ax.legend(loc="lower right")
show()

The imbalance and temporal controls remain visible in the prevalence columns. The
nominated model fell from 81.97% training AP to 1.14% on validation in the reference
run. The shared axis preserves the size of this gap. The separate AP chart shows
differences among the low validation scores.

## 14. Cohort and error analysis

For each cohort dimension, the table reconciles to the same validation sample and
frozen decision. Compare counts with alert rates divided by cohort size. A large group
can contribute a higher false-alert count even at a similar rate. The red marks show
false alerts as a share of all rows in that cohort.

In [ ]:
cohort_records = advanced["validation_cohorts"]

cohort = cohort_table(cohort_records)
display(format_table(cohort, {"Alert rate": "{:.1%}", "False alerts / all rows": "{:.1%}"}))
for _, dimension in cohort.groupby("Cohort"):
    columns = ["Rows", "Positive rows", "True alerts", "False alerts", "Missed churn"]
    assert dimension[columns].sum().tolist() == [
        chosen_metrics[name] for name in ("rows", "positives", "tp", "fp", "fn")
    ]
fig, axes = plt.subplots(2, 2, figsize=(11.5, 6))
cohort_titles = {
    "quarter": "Observation quarter",
    "episode_age": "Episode age",
    "billing_frequency": "Billing frequency",
    "usage_coverage": "Usage coverage",
}
for ax, (key, label) in zip(axes.flat, cohort_titles.items(), strict=True):
    subset = cohort[cohort["Cohort"] == key]
    ax.barh(subset["Group"], subset["Alert rate"], color="#255889")
    ax.scatter(
        subset["False alerts / all rows"],
        subset["Group"],
        color="#a74932",
        marker="|",
        s=115,
        label="False alerts / all rows",
    )
    ax.set(title=label, xlabel="Share of cohort observations", xlim=(0, 1))
    ax.xaxis.set_major_formatter(PercentFormatter(1))
fig.suptitle("Validation alert rate by cohort · fitted decision held fixed")
axes[0, 0].legend(loc="lower right", fontsize=8)
show()

In the retained run, seven events were in Q2 and one was in Q3. All eight were in
episodes aged 180–364 days. The nominee alerted 148 of 148 observations under 180 days,
all false alerts. The billing and usage groups are likewise too sparse for comparative
sensitivity claims. These patterns may partly reflect the fictional timing and reporting
rules. They do not identify a causal driver.

## 15. Ranking, operational use and the research nomination

Ranking orders observations, calibration compares scores with observed rates, and a
frozen threshold turns scores into alerts. Operational usefulness also requires useful
concentration of true alerts within a realistic queue. Here the predeclared rule
nominated the highest validation AP with a fixed simplicity tie-break. The nomination
preserves the fitted model and threshold. Its queue burden and weak ranking do not
establish business usefulness.

In [ ]:
chosen = winner["validation"]
lo, hi = advanced["validation_intervals"]["average_precision"]["interval_95"]
display(
    Markdown(
        f"**Frozen nomination** {winner['name']} at threshold **{winner['threshold']:.6f}**. "
        f"Validation AP **{chosen['average_precision']:.6f}** versus prior AP "
        f"**{reference_ap:.6f}**. The descriptive 95% account-bootstrap AP interval "
        f"was **{lo:.5f}–{hi:.5f}**."
    )
)
display(pd.DataFrame(advanced["validation_intervals"]).T)

The bootstrap conditions on the selected fit and observed accounts. It omits model
selection uncertainty. Resampling the same detected positives gives the degenerate
recall interval and supplies limited information about future recall.

## 16. Independent holdout status

**The seed-2027 reserve remains unscored in the retained experiment.**

It was drawn with a separate fictional namespace and Registry. The count-only audit
confirmed 19 positive accounts in that reserve. Validation had eight accounts against
the ten-account minimum, so confirmation was withheld. The archived Batch 09 evaluation
cannot substitute for this gate. The reserve shares a simulated catalogue, calendar and
behaviour assumptions. Even a scored reserve would only measure transfer within that
synthetic setting.

In [ ]:
gate = pd.DataFrame(design["support"]).T[["rows", "positives", "positive_customers", "prevalence"]]
gate["Required positive accounts"] = pd.Series(design["confirmation_support_minimum"])
display(format_table(gate, {"prevalence": "{:.2%}"}, na_rep="—"))
if advanced["holdout"] is None:
    failed = ", ".join(design["confirmation_support_failures"])
    display(Markdown(f"**Confirmation withheld** · support gate failed in {failed}."))
else:
    display(Markdown("This artefact records a scored reserve and differs from the reference run."))

## 17. Comparative conclusions

The prior dummy provides a simple AP reference. Logistic regression lowers the review
burden at its saved threshold while underperforming the reference on AP. The nominated
boosting candidate barely clears prevalence on AP, with weaker ROC-AUC, higher Brier
error and almost universal flagging. The training–validation contrast raises concerns
about stability on later accounts.

## 18. Final research assessment

The fixed experiment supports a transparent comparison under severe imbalance. Limited
event support leaves calibration, cohort reliability and independent generalisation
unresolved. The shared synthetic scenario also limits commercial inference. The retained
model remains an exploratory research candidate. The Project Owner accepted the Phase 06
engineering workflow with documented predictive limitations. Predictive and production approval
remain withheld, and the seed-2027 reserve remains unscored.